# Question 2 – Le Sudoku (5 points)

> **Mode d'emploi du book.**
> - Exécutez les cellules dans l'ordre (`Maj + Entrée`).
> - Les cellules marquées `# A COMPLETER` sont à compléter ; les cases **Votre réponse** sont à rédiger.

---

## Le problème

Une grille de Sudoku 9 × 9 est découpée en 9 blocs de 3 × 3. Il faut la compléter avec les chiffres 1 à 9 de sorte que
chaque **ligne**, chaque **colonne** et chaque **bloc** contienne chaque chiffre exactement une fois.

![Grille de sudoku à remplir](./images/sudokuARemplir.png)

Dans cette question, vous ne codez que le **modèle** et la **résolution** : les grilles et leur affichage sont fournis
par le fichier `outils_sudoku.py`.

---
## Les données (fournies)

`GRILLES` est un dictionnaire de grilles ; chaque grille est une liste de 9 listes de 9 entiers, 0 pour une case vide.
`afficher(grille)` affiche une grille.

In [ ]:
from outils_sudoku import GRILLES, afficher

print(list(GRILLES))
grille = GRILLES["cours"]
print(grille[0])          # la premiere ligne
afficher(grille)

---
## Réfléchir avant de coder

Rappel du carré magique (`MagicSquare-ort.py`) :

```python
c = [[model.new_int_var(1, n * n, f"c{i}.{j}") for j in range(n)] for i in range(n)]
model.add_all_different(v for ligne in c for v in ligne)
for i in range(n):
    model.add(sum(c[i]) == but)                                # lignes
    model.add(sum(c[k][i] for k in range(n)) == but)           # colonnes
```

**(R) Q2.1 –**
- Quelles sont les variables du Sudoku ? Combien y en a-t-il ? Quel est leur domaine ?
- Comment tenir compte des chiffres déjà donnés dans la grille ?
- Combien de contraintes « toutes différentes » faut-il au total ?
- Qu'est-ce qui se réutilise directement du carré magique, et qu'est-ce qui change ?

**Votre réponse à Q3.1 :**

*(double-cliquez ici pour écrire)*

---
## Étape 1 – Les variables

**Q2.2 –** Créez une variable par case, de domaine [1, 9], puis fixez les cases déjà remplies.
Vous devez obtenir **81 variables** et autant de contraintes que de chiffres donnés (**23** pour la grille du cours).

In [ ]:
from ortools.sat.python import cp_model

grille = GRILLES["cours"]
model = cp_model.CpModel()

x = ...                                      # A COMPLETER : 9 listes de 9 variables

# chiffres donnes
# A COMPLETER

print(len(model.proto.variables), "variables,", len(model.proto.constraints), "contraintes")

<details>
<summary><b>Indice</b></summary>

Comme dans le carré magique : `[[model.new_int_var(1, 9, f"x_{i}_{j}") for j in range(9)] for i in range(9)]`.
Pour une case non vide : `model.add(x[i][j] == grille[i][j])`.

</details>

---
## Étape 2 – Lignes et colonnes

**Q2.3 –** Ajoutez une contrainte `add_all_different` par ligne et par colonne.

In [ ]:
# A COMPLETER

print(len(model.proto.constraints), "contraintes")      # attendu : 23 + 18 = 41

<details>
<summary><b>Indice</b></summary>

La ligne `i` est déjà une liste : `x[i]`. La colonne `j` se construit comme dans le carré magique :
`[x[k][j] for k in range(9)]`.

</details>

---
## Étape 3 – Les blocs

**Q2.4 –** Ajoutez une contrainte `add_all_different` par bloc 3 × 3.

In [ ]:
# A COMPLETER

print(len(model.proto.constraints), "contraintes")      # attendu : 41 + 9 = 50

<details>
<summary><b>Indice</b></summary>

Le bloc (bi, bj), avec bi et bj entre 0 et 2, contient les cases `(3*bi + a, 3*bj + b)` pour a et b entre 0 et 2.
Deux boucles sur `bi` et `bj`, et une liste en compréhension sur `a` et `b`.

</details>

---
## Étape 4 – Résoudre

**Q2.5 –**
- Résolvez le modèle. Si une solution est trouvée, 
- construisez la grille solution (liste de 9 listes) et affichez-la avec `afficher(solution, grille)` :   
les chiffres trouvés par le solveur apparaissent entre crochets.

In [ ]:
solver = cp_model.CpSolver()
statut = solver.solve(model)
print(solver.status_name(statut))

# A COMPLETER : si une solution existe, construire la grille solution et l'afficher

<details>
<summary><b>Indice</b></summary>

`if statut in (cp_model.OPTIMAL, cp_model.FEASIBLE):` puis
`solution = [[solver.value(x[i][j]) for j in range(9)] for i in range(9)]`.

</details>

---
## Étape 5 – Une fonction réutilisable

**Q2.6 –** 
- Regroupez les étapes 1 à 4 dans une fonction `resoudre_sudoku(grille)`  
qui renvoie la grille solution, ou `None` s'il n'y a pas de solution. 
- La cellule de test l'applique à toutes les grilles fournies.

In [ ]:
def resoudre_sudoku(grille):
    # renvoie la grille solution (9 listes de 9 entiers), ou None s'il n'y a pas de solution
    model = cp_model.CpModel()
    # A COMPLETER (reprendre les etapes 1 a 4)
    ...


for nom, g in GRILLES.items():
    print("=====", nom)
    solution = resoudre_sudoku(g)
    if solution is None:
        print("Aucune solution")
    else:
        afficher(solution, g)

**(R) Q2.7 –** 
- Que répond votre fonction pour la grille `insoluble` ?  
- Regardez cette grille : contient-elle un chiffre en double dans une ligne, une colonne ou un bloc ? 
- Que signifie alors la réponse du solveur ?

**Votre réponse à Q2.7 :**

*(double-cliquez ici pour écrire)*

---
## Étape 6 – La solution est-elle unique ?

Une « vraie » grille de Sudoku a une **solution unique**. Pour le vérifier, on utilise la technique du cours
(`ColorZone-ort.py`) : on résout, on **interdit** la solution trouvée, on recommence.

```python
variables = [x[i][j] for i in range(9) for j in range(9)]
solutions = []
while len(solutions) < 2 and solver.solve(model) in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    sol = [solver.value(v) for v in variables]
    solutions.append(sol)
    model.add_forbidden_assignments(variables, [sol])   # interdit cette solution
```

**Q2.8 –** 
- Écrivez une fonction `nombre_de_solutions(grille)` qui renvoie 0, 1 ou 2 (2 signifiant « au moins deux »),  
et appliquez-la à toutes les grilles.

In [ ]:
def nombre_de_solutions(grille):
    # renvoie 0, 1 ou 2 (2 = au moins deux solutions)
    # A COMPLETER
    ...


for nom, g in GRILLES.items():
    print(nom, ":", nombre_de_solutions(g))

<details>
<summary><b>Indice</b></summary>

Recopiez le début de `resoudre_sudoku` (variables et contraintes), puis la boucle ci-dessus, et renvoyez
`len(solutions)`. Pourquoi s'arrêter à 2 solutions plutôt que de toutes les chercher ?

</details>

**(R) Q2.9 –** Quelle grille a plusieurs solutions ? Pourquoi suffit-il de chercher **deux** solutions pour savoir si
une grille est correcte ? Que se passerait-il si l'on cherchait toutes les solutions d'une grille presque vide ?

**Votre réponse à Q2.9 :**

*(double-cliquez ici pour écrire)*

---
## Pour aller plus loin (bonus)

- **Le solveur cherche-t-il ?** Affichez `solver.num_branches` après la résolution de chaque grille, puis recommencez
  avec `solver.parameters.cp_model_presolve = False`. Les grilles « difficiles » pour un humain le sont-elles pour le solveur ?

In [ ]:
# Espace libre pour les bonus